# 05 — Đánh giá bước 0 trên PADBen Task 4/5
Chỉ dùng checkpoint đã chốt. Không fine-tune và không chọn lại ngưỡng trên hai tập này.
Đây là đánh giá tác vụ/độ sâu trên nhóm nguồn giữ lại của PADBen; Task 1/2/3 đã góp vào train,
vì vậy không gọi đây là chứng minh tổng quát hóa sang một dataset hoàn toàn chưa thấy.

## Môi trường và đường dẫn
Chọn kernel Python đã cài dependency của repo. Notebook không tự cài/upgrade PyTorch.
Chạy các ô từ trên xuống. Có thể cấu hình `CATEX_PROJECT_ROOT`, `CATEX_DATA_DIR`, `CATEX_OUTPUT_ROOT` hoặc sửa ô dưới.
Không đưa `.env` lên notebook. Các notebook này không gọi Web Search hay LLM.


In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display

# Local: mở notebook từ repo hoặc notebooks/. Kaggle: sửa PROJECT_ROOT trỏ tới code đã giải nén.
PROJECT_ROOT = os.environ.get("CATEX_PROJECT_ROOT", "")
if PROJECT_ROOT:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "src/models/runtime.py").is_file()), None)
if ROOT is None or not (ROOT / "src/models/runtime.py").is_file():
    raise FileNotFoundError("Đặt PROJECT_ROOT thành thư mục chứa src/ và notebooks/.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

# Kaggle: sửa thành đường dẫn thư mục split hiển thị trong Input; không ghi vào Input.
DATA_DIR = Path(os.environ.get("CATEX_DATA_DIR", str(ROOT / "data/splits"))).resolve()
DEFAULT_OUTPUT = Path("/kaggle/working/catex") if Path("/kaggle").exists() else ROOT
OUTPUT_ROOT = Path(os.environ.get("CATEX_OUTPUT_ROOT", str(DEFAULT_OUTPUT))).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
from notebooks.workflow import inspect_csv, run_script, read_json, require_checkpoint
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Code:", ROOT, "\nData:", DATA_DIR, "\nOutput:", OUTPUT_ROOT)


In [ ]:
RUN_EVAL=False
BATCH_SIZE=32
# Sửa đường dẫn nếu checkpoint được gắn từ Kaggle Dataset/Notebook Output.
CHECKPOINT_DIR=OUTPUT_ROOT / "experiments/tang0_baseline_seed42/checkpoint"
REPORT_DIR=OUTPUT_ROOT / "evaluation/results/padben"
REPORT_DIR.mkdir(parents=True,exist_ok=True)
if RUN_EVAL:require_checkpoint(CHECKPOINT_DIR,0)


In [ ]:
for task in (4,5):
    path=DATA_DIR / f"padben_task{task}_test.csv"
    frame=pd.read_csv(path)
    assert set(frame.label)==({1} if task==4 else {0,1})
    assert frame["split"].eq("test").all()
    print(f"Task {task}: {len(frame):,} dòng")
    display(pd.crosstab(frame.original_task_label,frame.label))
    run_script(ROOT,"evaluation/cross_dataset_eval.py",
        ["--input",path,"--checkpoint",CHECKPOINT_DIR,"--batch-size",BATCH_SIZE,
         "--output",REPORT_DIR / f"task{task}.json"],
        enabled=RUN_EVAL,log_path=REPORT_DIR / f"task{task}.log")


In [ ]:
if RUN_EVAL:
    metrics=[]
    for task in (4,5):
        result=read_json(REPORT_DIR / f"task{task}.json")
        metrics.append({"task":task,**{k:result.get(k) for k in ["count","threshold","precision","recall","f1","roc_auc","single_class"]}})
        print(f"Task {task}: theo nhãn tác vụ gốc")
        display(pd.DataFrame(result.get("by_original_task_label",{})).T)
        fig,ax=plt.subplots(figsize=(4,3))
        matrix=np.array(result["confusion_matrix_labels_0_1"])
        ax.imshow(matrix,cmap="Blues")
        for i in range(2):
            for j in range(2):ax.text(j,i,str(matrix[i,j]),ha="center",va="center")
        ax.set(xticks=[0,1],yticks=[0,1],xlabel="Dự đoán",ylabel="Nhãn thật",title=f"Task {task}")
        fig.tight_layout();fig.savefig(REPORT_DIR / f"task{task}_confusion.png");plt.show()
    display(pd.DataFrame(metrics))
    pd.DataFrame(metrics).to_csv(REPORT_DIR / "summary.csv",index=False)
else:
    print("Chưa đánh giá; không hiển thị metric cũ như kết quả lần chạy này.")


## Diễn giải
- Task 4 toàn lớp 1: tập trung recall theo `original_task_label`; AUC/AP không xác định. Không dùng accuracy ở đây để chứng minh khả năng phân biệt human/AI.
- Task 5 có hai lớp: xem cả false positive và false negative, không chỉ accuracy.
- Mức giảm recall ở paraphrase sâu là kết quả cần báo cáo, không phải lý do chỉnh ngưỡng trên test.
- Các metric này chỉ đánh giá bước 0, chưa đo truy hồi web hay pipeline 5 tầng.
- Muốn đánh giá toàn bộ, dùng `evaluation/evaluate_full_pipeline.py` với báo cáo JSON và nhãn từng câu, theo `docs/PIPELINE.md`.